In [ ]:
# ======================================================
# Notebook: Bayesian Optimisation (Noisy Black-Box)
# Maximise unknown 2D log-likelihood function
# ======================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (N,2)
y = np.load("/mnt/data/initial_outputs.npy")     # (N,)

# Gaussian Process (handles noise)
kernel = ConstantKernel(1.0) * Matern(nu=2.5) + WhiteKernel()
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=True)
gp.fit(X, y)

# Expected Improvement (maximisation)
def expected_improvement(X_cand, X_sample, Y_sample, model, xi=0.01):
    mu, sigma = model.predict(X_cand, return_std=True)
    mu_sample = model.predict(X_sample)
    best = np.max(mu_sample)

    sigma = sigma.reshape(-1,1)
    mu = mu.reshape(-1,1)

    improvement = mu - best - xi
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma == 0.0] = 0.0
    return ei.ravel()

# Candidate grid
x_min, x_max = X[:,0].min(), X[:,0].max()
y_min, y_max = X[:,1].min(), X[:,1].max()

gx = np.linspace(x_min, x_max, 100)
gy = np.linspace(y_min, y_max, 100)
XX, YY = np.meshgrid(gx, gy)
X_grid = np.vstack([XX.ravel(), YY.ravel()]).T

# Acquisition
ei = expected_improvement(X_grid, X, y, gp)

# Next (10,2) inputs
top_idx = np.argsort(ei)[-10:]
next_points = X_grid[top_idx]

print("Next (10,2) inputs:")
print(next_points)

# Visualisation
mu, _ = gp.predict(X_grid, return_std=True)
plt.figure(figsize=(6,5))
plt.contourf(XX, YY, mu.reshape(XX.shape), levels=30)
plt.scatter(X[:,0], X[:,1])
plt.scatter(next_points[:,0], next_points[:,1], marker='x', s=100)
plt.show()